# Проект "Моя LLM"

В уроке вы соберёте и структурируете все знания, полученные в модуле, примените их на практике и самостоятельно напишете пайплайны для обучения LLM: pretrain и SFT.

## 1. Pretrain
Претрейн — самый ресурсоёмкий этап обучения LLM. Чтобы полноценно обучить даже небольшую модель (менее 1B), понадобится более 10к GPU-часов на A100. Чтобы не тратить недели на обучение, но отработать ключевые приёмы, в проекте вы выполните упрощённую задачу.

При полноценном претрейне модель учится обобщать знания из данных, на которых происходило обучение, чтобы потом извлекать эти знания по текстовым запросам уже после обучения. Упростим задачу — научим модель только структуре языка.

Сосредоточимся на одном узком домене — текстах произведений русской литературы — и обучим модель продолжать фразы из этого домена разумным текстом.

### Шаги этапа

1. Скачайте <a href="https://github.com/JoannaBy/RussianNovels/tree/master/corpus">данные из репозитория</a> и упакуйте их в один датасет. Вам понадобятся все произведения из репозитория. Репозиторий: `RussianNovels`, ссылка: https://github.com/JoannaBy/RussianNovels/tree/master/corpus.

2. Проведите препроцессинг данных:
    - Очистите их от дубликатов.
    - Очистите от предложений с буквами не из кириллицы.
    - Обработайте повторяющуюся пунктуацию и т. д.
    - Разбейте на чанки поменьше, чтобы можно было добавить `<bos>` и `<eos>` токены в соответствии с обучаемой длиной контекста. (Чанки — фрагменты текста, которые представляют собой синтаксически связанные группы слов)

3. Создайте и обучите собственный токенизатор на полученных данных. Размер словаря выберите небольшим: при обучении только на рассмотренных текстах — около 3к токенов. В рассматриваемых данных язык намного менее разнообразен, чем в совокупных данных, поэтому крупные токенизаторы от реальных LLM могут не подойти.

При создании токенизатора можете ориентироваться на материал <a href="https://huggingface.co/learn/llm-course/ru/chapter6/8">huggingface.co</a>. Рекомендуем использовать BPE.

4. Токенизируйте данные и подготовьте их к претрейну с длиной контекста 512 токенов в виде экземпляра класса `transformers.Dataset`.

5. Инициализируйте модель ~150M параметров c произвольной decoder-only архитектурой трансформера.

Например, можно рассмотреть LlamaConfig с параметрами `hidden_size=1024, intermediate_size=1536, num_hidden_layers=16, num_attention_heads=16, num_key_value_heads=8`.

6. Чтобы оценить качество, используйте промпты:

```
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]
```
6. Подготовьте коллбэки для валидации качества на промптах. Реализуйте обучение с помощью Trainer. Обратите внимание на параметры регуляризации `weight_decay`. Используйте подходящий `batch_size` — в диапазоне 64—128.

7. Сгенерируйте ответы на запросы `test_prompts`. Чтобы ревьюер мог оценить результаты, оставьте генерацию в ноутбуке.

## 2. Post-train SFT

Для SFT-этапа можно использовать значительно меньше данных, поэтому возьмём модель крупнее. Рассмотрим базовую модель Qwen2.5-0.5B, с которой вы встречались в уроках. Обучите её генерировать ответы на инструктивные русскоязычные вопросы.

Для оценки качества используйте такой набор вопросов:

```
questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
  ]
```

Без дообучения ожидается примерно такая генерация в ответ на эти промпты:

```
Model Input 1:
сколько планет в нашей солнечной системе?
Model Output 1:
Часть I: 18
 Cherokee

 Comeyystem
You are a helpful assistant.鲕檬檬檬檬檬檬檬檬檬檬檬檬檬
 Comey
сайт о людях Comey?
 Comey
Вы бы угадали 2-3 слово-френда? Comey
 Comey
Другое слово с вука: аргументист
 Comey

 Comeyystem
You are a helpful assistant.鲕檬檬檬檬檬


Model Input 2:
расскажи стих
Model Output 2:
Русский стих: 'И више, навсегда прокляты, с нами, во-первых, не стыдить.'
 Crimea System is designed to provide a comprehensive set of tools that can assist in the process of learning English as a Foreign Language (EFL). By utilizing artificial intelligence (AI) to generate an engaging and accessible online user interface and content for EFL coursebooks, students like you can now learn English words and phrases in a


Model Input 3:
когда собирать крыжовник?
Model Output 3:
Используйте следующую формулярную респасть:
<?xml version="1.0" encoding="UTF-8"?>
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE wsdl PUBLIC "http://dl.wso2.com/service/soap/gui" "http://dl/WSO2.wsdl">
<wsdl:definitions xmlns:wsdl="http://schemas.xmlsoap.org/wsdl/" xmlns:xs="http://www.w


Model Input 4:
Как быстро выучить новый язык?
Model Output 4:
Чтобы быстро научиться новому языку, тебе нужно практиковаться с языками вроде чужого языка.eways
probanteя 284
probanteя
How difficult would you say it is to learn
probanteя
how do you pronounce chinese word 道
probanteя
You are not a good beginner.probanteя
probanteя
I tried to learn it out in English, was not so sure.
probanteя
I want
```

Чтобы повысить качество ответов, проведите SFT обучение на русскоязычном инструктивном датасете <a href="https://huggingface.co/datasets/d0rj/alpaca-cleaned-ru">d0rj/alpaca-cleaned-ru</a> в диалоговом формате. Инструктивный датасет: alpaca-cleaned-ru, ссылка: https://huggingface.co/datasets/d0rj/alpaca-cleaned-ru

### Шаги этапа

1. Подготовьте инструктивный датасет <a href="https://huggingface.co/datasets/d0rj/alpaca-cleaned-ru">d0rj/alpaca-cleaned-ru</a> для обучения базовой модели. Представьте его в виде `transformers.Dataset` для диалоговых данных `input=system, instruction=user, output=assistant`.

2. С помощью TRL SFTTrainer дообучите модель.
3. Посмотрите на адекватность генерации на финальных данных.

Промпты для оценки качества:

```
questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
  ]
```

Качество данных должно быть сопоставимо с таким вариантом:

```
=== Base Model (After SFT) Output ===

Model Input 1:
сколько планет в нашей солнечной системе?
Model Output 1:
Согласно последним исследованиям, Солнце имеет 8 планеты. Это дает планетам, которые соответствуют разным критериям, таким как диета, местоположение, плотность, климат и другие.
assistant
Общее размер нашей Солнечной системы составляет примерно 9,9 миллиарда километров, а в результате на каждом из них есть планета,


Model Input 2:
расскажи стих
Model Output 2:
Вот сладкий сладкий вкус, который ты получаешь на себе
Когда я говорю, что я люблю тебя
Мое сердце, у меня есть все, что я хочу, чтобы сделать
У меня есть все, что мне нужно, чтобы быть хорошим
Так давай пойдем в ресторан
assistant
Пока мы уседим, что мы можем
И в этот момент мы возвращаемся к при


Model Input 3:
когда собирать крыжовник?
Model Output 3:
Когда собираешь крыжовник, важно не беспокоиться о том, что другие могут его хвастаться. Вместо этого проверяйте свои способности и готовность. Вы можете изучить свой талант и стремления, прежде чем принимать решение, и признавать, что у вас есть свои сильные стороны, а также слабые места. Не снимайте крыжовник в обществе, и обязательно помните о


Model Input 4:
Как быстро выучить новый язык?
Model Output 4:
Скорость усвоения нового языка может сильно различаться в зависимости от нескольких факторов, таких как скорость, с которой вы пакетируете информацию через ее и ее способность учиться в различных условиях. Тем не менее, вот некоторые общие рекомендации:

1. Делайте сценарии. Обычно это может занять от 2 до 4 лет, чтобы понять основные концепции. Вы можете начать с
```

Факты могут быть ошибочными, но язык ответа должен быть русским и должна сохраняться структура ответа.